In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = [12, 6]

# Display all columns
pd.set_option('display.max_columns', None)

In [ ]:
# Define a function to load data files
def load_data(file_path):
    """Load data from a CSV file with appropriate parsing"""
    try:
        # Attempt to load the file
        if 'Unemployment_Rate' in file_path:
            # Special handling for unemployment data which has a different structure
            df = pd.read_csv(file_path)
            # Extract year and month from Label column
            df[['Year', 'Month']] = df['Label'].str.split(' ', expand=True)
            # Convert to numeric
            df['Year'] = pd.to_numeric(df['Year'])
            return df
        elif 'zhvi_home_values.csv' in file_path:
            # Special handling for ZHVI data to filter for DC
            df = pd.read_csv(file_path)
            
            # Filter for DC Metro area
            # Try multiple approaches to find DC data
            dc_df = df[df['RegionName'] == 'Washington, DC']
            
            # If no matches, try looking for Washington in DC state
            if len(dc_df) == 0:
                dc_df = df[(df['RegionName'].str.contains('Washington', case=False)) & 
                           (df['StateName'] == 'DC')]
            
            # If still no matches, try Washington metro area
            if len(dc_df) == 0:
                dc_df = df[df['RegionName'] == 'Washington']
                
            # If all fails, just return the first DC entry
            if len(dc_df) == 0:
                dc_df = df[df['StateName'] == 'DC'].head(1)
                
            return dc_df
        else:
            # Standard CSV loading for other files
            df = pd.read_csv(file_path)
            return df
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Load all datasets
data_dir = "Data/"
datasets = {}

# Divorce Rate
divorce_rate_df = load_data(os.path.join(data_dir, "DC_Divorce_Rate.csv"))
# Marriage Rate
marriage_rate_df = load_data(os.path.join(data_dir, "DC_Marriage_Rate.csv"))
# Median Household Income
income_df = load_data(os.path.join(data_dir, "DC_Median_Household_Income_Annually.csv"))
# Population
population_df = load_data(os.path.join(data_dir, "DC_Population.csv"))
# Poverty Rate
poverty_df = load_data(os.path.join(data_dir, "DC_Poverty_Rate.csv"))
# Unemployment Rate (monthly)
unemployment_df = load_data(os.path.join(data_dir, "DC_Unemployment_Rate.csv"))
# Consumer Price Index (monthly)
cpi_df = load_data(os.path.join(data_dir, "DMV_CPI.csv"))
# GDP Growth (quarterly)
gdp_df = load_data(os.path.join(data_dir, "GDP_Growth.csv"))
# Interest Rates (monthly)
interest_df = load_data(os.path.join(data_dir, "Interest_Rates.csv"))
# Mortgage Rate (weekly)
mortgage_df = load_data(os.path.join(data_dir, "Mortgage_Rate.csv"))
# ZHVI (Zillow Home Value Index) - if available
try:
    zhvi_df = load_data(os.path.join(data_dir, "zhvi_home_values.csv"))
    print("Successfully loaded ZHVI data")
except:
    print("Note: ZHVI data file not found. Continuing without it.")
    zhvi_df = None

# 2. Examine Data Structure
# =========================

# Function to display basic info about each dataframe
def examine_dataframe(name, df):
    """Display basic information about a dataframe"""
    if df is not None:
        print(f"\n=== {name} ===")
        print(f"Shape: {df.shape}")
        print("\nSample data:")
        print(df.head())
        print("\nData types:")
        print(df.dtypes)
        print("\nNull values:")
        print(df.isnull().sum())
        print("=" * 50)
    else:
        print(f"Dataset {name} could not be loaded.")

# Examine each dataset
examine_dataframe("Divorce Rate", divorce_rate_df)
examine_dataframe("Marriage Rate", marriage_rate_df)
examine_dataframe("Median Household Income", income_df)
examine_dataframe("Population", population_df)
examine_dataframe("Poverty Rate", poverty_df)
examine_dataframe("Unemployment Rate", unemployment_df)
examine_dataframe("Consumer Price Index", cpi_df)
examine_dataframe("GDP Growth", gdp_df)
examine_dataframe("Interest Rates", interest_df)
examine_dataframe("Mortgage Rate", mortgage_df)
if zhvi_df is not None:
    examine_dataframe("ZHVI", zhvi_df)

# 3. Data Preprocessing
# =====================

# Let's first fix the divorce and marriage rate dataframes
# These appear to have years as columns, need to pivot

# Process Divorce Rate
def process_divorce_rate():
    if divorce_rate_df is not None:
        # Assuming first row contains the data values
        years = divorce_rate_df.columns.tolist()
        rates = divorce_rate_df.iloc[0].tolist()
        
        # Create a new dataframe with correct structure
        df = pd.DataFrame({
            'Year': [int(year) for year in years],
            'Divorce_Rate': rates
        })
        return df
    return None

# Process Marriage Rate
def process_marriage_rate():
    if marriage_rate_df is not None:
        # Assuming first row contains the data values
        years = marriage_rate_df.columns.tolist()
        rates = marriage_rate_df.iloc[0].tolist()
        
        # Create a new dataframe with correct structure
        df = pd.DataFrame({
            'Year': [int(year) for year in years],
            'Marriage_Rate': rates
        })
        return df
    return None

# Process Population
def process_population():
    if population_df is not None:
        # Extract the year from observation_date
        population_df['Year'] = pd.to_datetime(population_df['observation_date']).dt.year
        
        # Rename for clarity
        df = population_df.rename(columns={'DCPOP': 'Population'})
        
        # Select relevant columns
        df = df[['Year', 'Population']]
        return df
    return None

# Process Income data
def process_income():
    if income_df is not None:
        # Extract the year from observation_date
        income_df['Year'] = pd.to_datetime(income_df['observation_date']).dt.year
        
        # Rename for clarity
        df = income_df.rename(columns={'MEHOINUSDCA646N': 'Median_Household_Income'})
        
        # Select relevant columns
        df = df[['Year', 'Median_Household_Income']]
        return df
    return None

# Process Poverty Rate
def process_poverty():
    if poverty_df is not None:
        # Convert Year to integer
        poverty_df['Year'] = poverty_df['Year'].astype(int)
        
        # Rename for clarity
        df = poverty_df.rename(columns={'Poverty Rate (%)': 'Poverty_Rate'})
        
        return df
    return None

# Process monthly unemployment to annual averages
def process_unemployment():
    if unemployment_df is not None:
        # Group by year and calculate the annual average
        df = unemployment_df.groupby('Year')['Value'].mean().reset_index()
        
        # Rename for clarity
        df = df.rename(columns={'Value': 'Unemployment_Rate'})
        
        return df
    return None

# Process CPI (convert monthly to annual average)
def process_cpi():
    if cpi_df is not None:
        # Melt the dataframe to have months as rows
        df_melted = pd.melt(cpi_df, id_vars=['Year'], 
                          value_vars=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                                     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
                          var_name='Month', value_name='CPI')
        
        # Group by year and calculate the annual average
        df = df_melted.groupby('Year')['CPI'].mean().reset_index()
        
        # Convert Year to integer
        df['Year'] = df['Year'].astype(int)
        
        return df
    return None

# Process Interest Rates (convert monthly to annual average)
def process_interest_rates():
    if interest_df is not None:
        # Extract the year from observation_date
        interest_df['Year'] = pd.to_datetime(interest_df['observation_date']).dt.year
        
        # Group by year and calculate the annual average
        df = interest_df.groupby('Year')['FEDFUNDS'].mean().reset_index()
        
        # Rename for clarity
        df = df.rename(columns={'FEDFUNDS': 'Interest_Rate'})
        
        return df
    return None

# Process Mortgage Rates (convert weekly to annual average)
def process_mortgage_rates():
    if mortgage_df is not None:
        # Extract the year from Week column (assuming format like '12/31/1999')
        mortgage_df['Year'] = pd.to_datetime(mortgage_df['Week'], format='%m/%d/%Y').dt.year
        
        # Group by year and calculate the annual average
        df = mortgage_df.groupby('Year')['FRM'].mean().reset_index()
        
        # Rename for clarity
        df = df.rename(columns={'FRM': 'Mortgage_Rate'})
        
        return df
    return None

# Process GDP Growth (convert quarterly to annual average) - FIXED
def process_gdp():
    if gdp_df is not None:
        # FIXED: The GDP data only has 2 rows (index 0 and 1)
        # Row 0 contains quarters (Q1, Q2, etc)
        # Row 1 contains the actual values
        
        # Extract years from column names and growth values
        years = []
        growth_values = []
        
        for col in gdp_df.columns:
            try:
                # Extract the year part before any decimal point
                year = int(float(col))
                if year not in years:
                    # For each new year, calculate the average of its quarters
                    year_cols = [c for c in gdp_df.columns if str(c).startswith(str(year))]
                    year_values = [float(gdp_df.iloc[1, gdp_df.columns.get_loc(c)]) for c in year_cols]
                    years.append(year)
                    growth_values.append(sum(year_values) / len(year_values))
            except (ValueError, TypeError):
                # Skip columns that aren't years
                continue
        
        # Create a dataframe with years and average growth values
        df = pd.DataFrame({
            'Year': years,
            'GDP_Growth': growth_values
        })
        
        # Sort by year
        df = df.sort_values('Year')
        
        return df
    return None

# Process ZHVI data (if available)
def process_zhvi():
    if zhvi_df is not None:
        # Get all date columns (those that start with numbers)
        date_columns = [col for col in zhvi_df.columns if str(col)[0].isdigit()]
        
        # Convert the wide format to long format
        df_long = pd.melt(
            zhvi_df,
            id_vars=['RegionID', 'RegionName', 'RegionType', 'StateName'],
            value_vars=date_columns,
            var_name='Date',
            value_name='ZHVI'
        )
        
        # Convert the date string to datetime
        df_long['Date'] = pd.to_datetime(df_long['Date'])
        
        # Extract year
        df_long['Year'] = df_long['Date'].dt.year
        
        # Group by year to get annual average
        df_annual = df_long.groupby('Year')['ZHVI'].mean().reset_index()
        
        return df_annual
    return None

# Apply preprocessing to create cleaned dataframes
divorce_clean = process_divorce_rate()
marriage_clean = process_marriage_rate()
population_clean = process_population()
income_clean = process_income()
poverty_clean = process_poverty()
unemployment_clean = process_unemployment()
cpi_clean = process_cpi()
interest_clean = process_interest_rates()
mortgage_clean = process_mortgage_rates()
gdp_clean = process_gdp()
zhvi_clean = process_zhvi()

# Display sample of processed dataframes
for name, df in [
    ("Processed Divorce Rate", divorce_clean),
    ("Processed Marriage Rate", marriage_clean),
    ("Processed Population", population_clean),
    ("Processed Income", income_clean),
    ("Processed Poverty Rate", poverty_clean),
    ("Processed Unemployment Rate", unemployment_clean),
    ("Processed CPI", cpi_clean),
    ("Processed Interest Rates", interest_clean),
    ("Processed Mortgage Rates", mortgage_clean),
    ("Processed GDP Growth", gdp_clean),
    ("Processed ZHVI", zhvi_clean) if zhvi_clean is not None else (None, None)
]:
    if df is not None and name is not None:
        print(f"\n=== {name} ===")
        print(df.head())
        print("-" * 40)

# 4. Create Main Dataframe
# ========================

# Merge all dataframes based on Year
def create_main_dataframe():
    # Start with the first dataframe (divorce rate)
    main_df = divorce_clean.copy()
    
    # List of dataframes to merge
    dfs_to_merge = [
        marriage_clean,
        population_clean,
        income_clean, 
        poverty_clean,
        unemployment_clean,
        cpi_clean,
        interest_clean,
        mortgage_clean,
        gdp_clean
    ]
    
    # Add ZHVI if available
    if zhvi_clean is not None:
        dfs_to_merge.append(zhvi_clean)
    
    # Merge all dataframes on Year
    for df in dfs_to_merge:
        if df is not None:
            main_df = pd.merge(main_df, df, on='Year', how='outer')
    
    # Sort by year
    main_df = main_df.sort_values('Year')
    
    return main_df

# Create the main dataframe
main_df = create_main_dataframe()

# Display the main dataframe
print("\n=== Main Dataframe ===")
print(main_df.head(10))
print("-" * 40)
print("Shape:", main_df.shape)
print("-" * 40)
print("Missing values:")
print(main_df.isnull().sum())
print("-" * 40)

# 5. Basic Exploratory Analysis
# =============================

# Check data range
print("\nData range (years covered):")
print(f"Min Year: {main_df['Year'].min()}, Max Year: {main_df['Year'].max()}")

# Summary statistics
print("\nSummary statistics:")
print(main_df.describe())

# Create output directory structure
import os
output_dir = os.path.join('outputs', 'EDA')
os.makedirs(output_dir, exist_ok=True)

# Save the main dataframe for further analysis
main_df.to_csv(os.path.join(output_dir, 'dc_economic_main_dataframe_yearly.csv'), index=False)
print(f"\nMain dataframe saved to '{os.path.join(output_dir, 'dc_economic_main_dataframe_yearly.csv')}'")

# 6. Initial Visualizations
# =========================

# Set up plotting
plt.figure(figsize=(15, 10))

# Plot median household income over time
plt.subplot(2, 2, 1)
plt.plot(main_df['Year'], main_df['Median_Household_Income'], marker='o', linewidth=2)
plt.title('DC Median Household Income (2000-2023)')
plt.xlabel('Year')
plt.ylabel('Median Income ($)')
plt.grid(True)

# Plot population over time
plt.subplot(2, 2, 2)
plt.plot(main_df['Year'], main_df['Population'], marker='o', linewidth=2, color='green')
plt.title('DC Population (2000-2024)')
plt.xlabel('Year')
plt.ylabel('Population')
plt.grid(True)

# Plot unemployment rate over time
plt.subplot(2, 2, 3)
plt.plot(main_df['Year'], main_df['Unemployment_Rate'], marker='o', linewidth=2, color='red')
plt.title('DC Unemployment Rate (2000-2024)')
plt.xlabel('Year')
plt.ylabel('Unemployment Rate (%)')
plt.grid(True)

# Plot poverty rate over time
plt.subplot(2, 2, 4)
plt.plot(main_df['Year'], main_df['Poverty_Rate'], marker='o', linewidth=2, color='purple')
plt.title('DC Poverty Rate (2000-2023)')
plt.xlabel('Year')
plt.ylabel('Poverty Rate (%)')
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'initial_trends_yearly.png'))
plt.show()

# Plot marriage and divorce rates
plt.figure(figsize=(12, 6))
plt.plot(main_df['Year'], main_df['Marriage_Rate'], marker='o', linewidth=2, label='Marriage Rate')
plt.plot(main_df['Year'], main_df['Divorce_Rate'], marker='o', linewidth=2, label='Divorce Rate')
plt.title('DC Marriage and Divorce Rates (2000-2022)')
plt.xlabel('Year')
plt.ylabel('Rate (per 1,000 population)')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(output_dir, 'marriage_divorce_trends_yearly.png'))
plt.show()

# Plot interest rates and mortgage rates
plt.figure(figsize=(12, 6))
plt.plot(main_df['Year'], main_df['Interest_Rate'], marker='o', linewidth=2, label='Federal Funds Rate')
plt.plot(main_df['Year'], main_df['Mortgage_Rate'], marker='o', linewidth=2, label='30-Year Fixed Mortgage Rate')
plt.title('Interest Rates and Mortgage Rates (2000-2024)')
plt.xlabel('Year')
plt.ylabel('Rate (%)')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(output_dir, 'interest_mortgage_trends_yearly.png'))
plt.show()

# Plot ZHVI if available
if 'ZHVI' in main_df.columns:
    plt.figure(figsize=(12, 6))
    plt.plot(main_df['Year'], main_df['ZHVI'], marker='o', linewidth=2, color='darkblue')
    plt.title('DC Home Value Index (ZHVI) Over Time')
    plt.xlabel('Year')
    plt.ylabel('Home Value Index')
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'zhvi_trend_yearly.png'))
    plt.show()
    
    # Also plot ZHVI against other key variables
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # ZHVI vs Income
    axes[0, 0].scatter(main_df['Median_Household_Income'], main_df['ZHVI'])
    axes[0, 0].set_title('ZHVI vs Median Household Income')
    axes[0, 0].set_xlabel('Median Household Income ($)')
    axes[0, 0].set_ylabel('Home Value Index')
    
    # ZHVI vs Mortgage Rate
    axes[0, 1].scatter(main_df['Mortgage_Rate'], main_df['ZHVI'])
    axes[0, 1].set_title('ZHVI vs Mortgage Rate')
    axes[0, 1].set_xlabel('Mortgage Rate (%)')
    axes[0, 1].set_ylabel('Home Value Index')
    
    # ZHVI vs Unemployment
    axes[1, 0].scatter(main_df['Unemployment_Rate'], main_df['ZHVI'])
    axes[1, 0].set_title('ZHVI vs Unemployment Rate')
    axes[1, 0].set_xlabel('Unemployment Rate (%)')
    axes[1, 0].set_ylabel('Home Value Index')
    
    # ZHVI vs Population
    axes[1, 1].scatter(main_df['Population'], main_df['ZHVI'])
    axes[1, 1].set_title('ZHVI vs Population')
    axes[1, 1].set_xlabel('Population')
    axes[1, 1].set_ylabel('Home Value Index')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'zhvi_correlations_yearly.png'))
    plt.show()

# 7. Correlation Analysis
# =======================

# Calculate the correlation matrix
correlation_matrix = main_df.corr()

# Plot the correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5, fmt=".2f")
plt.title('Correlation Matrix of DC Economic Indicators')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'correlation_matrix_yearly.png'))
plt.show()

# 8. Next Steps
# =============
# - Handle missing values (interpolation or other methods)
# - Perform more detailed analysis on specific relationships
# - Create more advanced visualizations
# - Conduct statistical tests for relationships
# - Consider time-series analysis techniques
# - Build regression models with ZHVI as dependent variable